In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


import matplotlib
import matplotlib.pyplot as plt

#SKLEARN
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier # Import Decision Tree Classifier
from sklearn.model_selection import train_test_split # Import train_test_split function
from sklearn import metrics #Import scikit-learn metrics module for accuracy calculation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn import feature_extraction, linear_model, model_selection, preprocessing
#OTHER LIBRARIES
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
import string
import re
import nltk





# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 5GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
print('Loading Successful.')

/kaggle/input/nlp-getting-started/train.csv
/kaggle/input/nlp-getting-started/test.csv
/kaggle/input/nlp-getting-started/sample_submission.csv
Loading Successful.


In [2]:
test_data  = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
train_data = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
sub = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
print('Loading Successful.')

Loading Successful.


In [3]:
train_data.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [4]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


In [5]:
train_data_no_duplicates = train_data.drop_duplicates(subset=['text'])

In [6]:
x1 = len(train_data) #=7613
x2 = len(set(train_data['text'])) #=7498
number_of_duplicated_records = (x1 - x2) 
print('Number of duplicated records is:',number_of_duplicated_records)

Number of duplicated records is: 110


In [7]:
train_data_no_duplicates.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [8]:
train_data_no_duplicates.shape

(7503, 5)

In [9]:
x1 = len(train_data_no_duplicates) #=7613
x2 = len(set(train_data_no_duplicates['text'])) #=7498
number_of_duplicated_records = (x1 - x2) 
print('Number of duplicated records is:',number_of_duplicated_records)

Number of duplicated records is: 0


In [10]:
def remove_punct(text):
    text_nopunct = ''.join([char for char in text if char not in string.punctuation])
    return text_nopunct

train_data_no_duplicates['text'] = train_data_no_duplicates['text'].apply(lambda x: remove_punct(x))
#this is how you can drop any column. --> train_data.drop('text_clean', axis=1, inplace=True)
train_data_no_duplicates.head()

/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this earthquake Ma...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask Canada,1
2,5,NaN,NaN,All residents asked to shelter in place are be...,1
3,6,NaN,NaN,13000 people receive wildfires evacuation orde...,1
4,7,NaN,NaN,Just got sent this photo from Ruby Alaska as s...,1


In [11]:
#train_data_no_duplicates=train_data_no_duplicates.drop('location',1)

#test_data=test_data.drop('location',1)

In [12]:
train_data_no_duplicates.head()


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this earthquake Ma...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask Canada,1
2,5,NaN,NaN,All residents asked to shelter in place are be...,1
3,6,NaN,NaN,13000 people receive wildfires evacuation orde...,1
4,7,NaN,NaN,Just got sent this photo from Ruby Alaska as s...,1


In [13]:

vectorizer = CountVectorizer(analyzer='word', binary=True)
vectorizer.fit(train_data_no_duplicates['text'])
X = vectorizer.transform(train_data_no_duplicates['text']).todense()
y = train_data_no_duplicates['target'].values
X.shape, y.shape


((7503, 22394), (7503,))

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2020)


model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [15]:
y_pred = model.predict(X_test)

f1score = f1_score(y_test, y_pred)
print(f"Model Score: {f1score * 100} %")

Model Score: 75.3694581280788 %


In [16]:
tweets_test = test_data['text']
test_X = vectorizer.transform(tweets_test).todense()
test_X.shape

(3263, 22394)

In [17]:
lr_pred = model.predict(test_X)

In [18]:
sub['target'] = lr_pred
sub.to_csv("submission.csv", index=False)
sub.head()

,id,target
0,0,1
1,2,0
2,3,1
3,9,0
4,11,1
